In [1]:
import os
import json
import time
from openai import OpenAI

API_KEY = "sk-proj-xxxxxxxxxxxxxxxxx" 
client = OpenAI(api_key=API_KEY)

In [2]:
# to avoid API failures

def generate_completion(model, system_prompt, user_content):
    
    """Helper to handle API calls with basic retry logic"""
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],

                # for stable reasoning
                temperature=0.0
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2) # Wait 2s before retry
            else:
                print(f"  [!] Failed after {max_retries} attempts: {e}")
                return None

In [3]:
# API connection test
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say hello in one word"
)

print(response.output_text)

Hello!


In [4]:
# AMI (Full Training Set)
INPUT_FILE = "phase1_ami_clean_train.jsonl"
OUTPUT_FILE = "phase1b_ami_train_reasoning.jsonl"
MODEL = "gpt-4.1-mini"

SYSTEM_PROMPT = """You are a Meeting Scribe Agent.
Task: Summarize the meeting transcript.
Output Format:
Thought: <Identify the key speakers, the main decision made, and action items.>
Summary: <The final abstractive summary>
"""

In [5]:
print(f"Starting AMI Processing ({INPUT_FILE})...")

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f_in, open(OUTPUT_FILE, 'w') as f_out:
        lines = f_in.readlines()
        print(f"  -> Found {len(lines)} meetings to process.")
        
        for i, line in enumerate(lines):
            data = json.loads(line)
            
            output = generate_completion(MODEL, SYSTEM_PROMPT, data['input'])
            
            if output:
                data['output'] = output
                f_out.write(json.dumps(data) + "\n")
            
            if (i+1) % 10 == 0: 
                print(f"  [AMI] Processed {i+1}/{len(lines)}")

    print(f"AMI Complete! Saved to {OUTPUT_FILE}")
else:
    print(f"Error: {INPUT_FILE} not found.")

Starting AMI Processing (phase1_ami_clean_train.jsonl)...
  -> Found 145 meetings to process.
  [AMI] Processed 10/145
  [AMI] Processed 20/145
  [AMI] Processed 30/145
  [AMI] Processed 40/145
  [AMI] Processed 50/145
  [AMI] Processed 60/145
  [AMI] Processed 70/145
  [AMI] Processed 80/145
  [AMI] Processed 90/145
  [AMI] Processed 100/145
  [AMI] Processed 110/145
  [AMI] Processed 120/145
  [AMI] Processed 130/145
  [AMI] Processed 140/145
AMI Complete! Saved to phase1b_ami_train_reasoning.jsonl
